In [1]:
import ast
import pandas as pd
from pathlib import Path

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

vector_results = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "vector_retrieval_results.csv")
graph_results = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "graph_retrieval_results.csv")
hybrid_results = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "hybrid_retrieval_results.csv")
evaluation_summary = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "retrieval_evaluation_summary.csv")

evaluation_summary.head()

,query_artist,vector_top1,vector_score,hybrid_top1,hybrid_score,top1_changed,shared_member,shared_tag,graph_bonus,interpretation
0,Soundgarden,Pearl Jam,0.740760,Nirvana,1.011961,True,1.0,6.0,0.300000,Hybrid changed the top result due to shared-me...
1,Nirvana,Pearl Jam,0.754655,Soundgarden,0.997676,True,1.0,6.0,0.285714,Hybrid changed the top result due to shared-me...
2,Deftones,Blur,0.691868,Soundgarden,0.781798,True,0.0,2.0,0.100000,Hybrid changed the top result due to shared-ta...
3,Pearl Jam,Blur,0.760920,Soundgarden,0.997903,True,1.0,4.0,0.257143,Hybrid changed the top result due to shared-me...
4,The Smashing Pumpkins,Blur,0.696780,Blur,0.796780,False,0.0,5.0,0.100000,Vector and hybrid agree on the top result.


In [3]:
def parse_evidence(value):
    if pd.isna(value):
        return []
    
    if isinstance(value, list):
        return value
    
    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return [value]

graph_results["evidence_parsed"] = graph_results["evidence"].apply(parse_evidence)

graph_results.head()

,query_artist,rank,retrieved_artist,graph_score,evidence_type,evidence,evidence_parsed
0,Blur,1,Pearl Jam,7,shared_tag,"['alternative rock', 'grunge', 'rock', 'art ro...","[alternative rock, grunge, rock, art rock, exp..."
1,Blur,2,The Smashing Pumpkins,5,shared_tag,"['alternative rock', 'grunge', 'rock', 'shoega...","[alternative rock, grunge, rock, shoegaze, neo..."
2,Blur,3,Nirvana,4,shared_tag,"['alternative rock', 'grunge', 'noise rock', '...","[alternative rock, grunge, noise rock, rock]"
3,Deftones,1,Soundgarden,2,shared_tag,"['alternative metal', 'alternative rock']","[alternative metal, alternative rock]"
4,Deftones,2,Nirvana,2,shared_tag,"['alternative metal', 'alternative rock']","[alternative metal, alternative rock]"


In [4]:
def get_graph_evidence(query_artist, retrieved_artist):
    evidence_rows = graph_results[
        (graph_results["query_artist"] == query_artist) &
        (graph_results["retrieved_artist"] == retrieved_artist)
    ]

    if evidence_rows.empty:
        return {
            "shared_members": [],
            "shared_tags": []
        }

    shared_members = []
    shared_tags = []

    for _, row in evidence_rows.iterrows():
        if row["evidence_type"] == "shared_member":
            shared_members.extend(row["evidence_parsed"])
        elif row["evidence_type"] == "shared_tag":
            shared_tags.extend(row["evidence_parsed"])

    return {
        "shared_members": shared_members,
        "shared_tags": shared_tags
    }

In [5]:
get_graph_evidence("Pearl Jam", "Soundgarden")

{'shared_members': ['Matt Cameron'],
 'shared_tags': ['alternative rock', 'grunge', 'hard rock', 'usa']}

In [6]:
get_graph_evidence("Nirvana", "Soundgarden")

{'shared_members': ['Jason Everman'],
 'shared_tags': ['alternative metal',
  'alternative rock',
  'american',
  'grunge',
  'sludge metal',
  'usa']}

In [7]:
get_graph_evidence("Blur", "Soundgarden")

{'shared_members': [], 'shared_tags': []}

In [18]:
# def generate_explanation(query_artist, retrieved_artist):
#     vector_row = vector_results[
#         (vector_results["query_artist"] == query_artist) &
#         (vector_results["retrieved_artist"] == retrieved_artist)
#     ]

#     hybrid_row = hybrid_results[
#         (hybrid_results["query_artist"] == query_artist) &
#         (hybrid_results["retrieved_artist"] == retrieved_artist)
#     ]

#     evidence = get_graph_evidence(query_artist, retrieved_artist)

#     explanation_parts = []

#     explanation_parts.append(
#         f"{retrieved_artist} was retrieved for {query_artist}."
#     )
    
#     if vector_row.empty and hybrid_row.empty and not evidence["shared_members"] and not evidence["shared_tags"]:
#     explanation_parts.append(
#         "This pair does not appear in the saved top-k vector, graph, or hybrid retrieval results for this query direction."
#     )
#     explanation_parts.append(
#         "This does not necessarily mean the artists have no similarity or relationship; it only means the pair was not retrieved in the current saved top-k results."
#     )
#     return " ".join(explanation_parts)

#     if not vector_row.empty:
#         vector_score = vector_row.iloc[0]["similarity_score"]
#         explanation_parts.append(
#             f"In the vector retriever, this pair has a cosine similarity score of {vector_score:.3f}, based on artist embedding text."
#         )

#     if evidence["shared_members"]:
#         members = ", ".join(evidence["shared_members"])
#         explanation_parts.append(
#             f"The graph retriever also found shared-member evidence: {members} connects {query_artist} and {retrieved_artist}."
#         )

#     if evidence["shared_tags"]:
#         tags = ", ".join(evidence["shared_tags"][:5])
#         explanation_parts.append(
#             f"The graph also found shared tag evidence, including: {tags}."
#         )

#     if not hybrid_row.empty:
#         hybrid_score = hybrid_row.iloc[0]["hybrid_score"]
#         explanation_parts.append(
#             f"The hybrid retriever combines vector similarity with graph evidence, giving this pair a hybrid score of {hybrid_score:.3f}."
#         )

#     if evidence["shared_members"]:
#         explanation_parts.append(
#             "This is a relationship-based discovery because the connection is supported by explicit historical graph evidence."
#         )
#     elif evidence["shared_tags"]:
#         explanation_parts.append(
#             "This is mainly a style or metadata-based discovery because the graph support comes from shared tags."
#         )

#     return " ".join(explanation_parts)

In [20]:
def generate_explanation(query_artist, retrieved_artist):
    
    vector_row = vector_results[
        (vector_results["query_artist"] == query_artist) &
        (vector_results["retrieved_artist"] == retrieved_artist)
    ]

    hybrid_row = hybrid_results[
        (hybrid_results["query_artist"] == query_artist) &
        (hybrid_results["retrieved_artist"] == retrieved_artist)
    ]

    evidence = get_graph_evidence(query_artist, retrieved_artist)

    explanation_parts = []
    explanation_parts.append(f"{retrieved_artist} was retrieved for {query_artist}.")
    
    if (vector_row.empty and hybrid_row.empty and 
        not evidence["shared_members"] and not evidence["shared_tags"]):
        
        explanation_parts.append(
            "This pair does not appear in the saved top-k vector, graph, or hybrid retrieval results for this query direction."
        )
        explanation_parts.append(
            "This does not necessarily mean the artists have no similarity or relationship; it only means the pair was not retrieved in the current saved top-k results."
        )
        return " ".join(explanation_parts)

    if not vector_row.empty:
        vector_score = vector_row.iloc[0]["similarity_score"]
        explanation_parts.append(
            f"In the vector retriever, this pair has a cosine similarity score of {vector_score:.3f}, based on artist embedding text."
        )

    if evidence["shared_members"]:
        members = ", ".join(evidence["shared_members"])
        explanation_parts.append(
            f"The graph retriever also found shared-member evidence: {members} connects {query_artist} and {retrieved_artist}."
        )

    if evidence["shared_tags"]:
        tags = ", ".join(evidence["shared_tags"][:5])
        explanation_parts.append(
            f"The graph also found shared tag evidence, including: {tags}."
        )

    if not hybrid_row.empty:
        hybrid_score = hybrid_row.iloc[0]["hybrid_score"]
        explanation_parts.append(
            f"The hybrid retriever combines vector similarity with graph evidence, giving this pair a hybrid score of {hybrid_score:.3f}."
        )

    if evidence["shared_members"]:
        explanation_parts.append(
            "This is a relationship-based discovery because the connection is supported by explicit historical graph evidence."
        )
    elif evidence["shared_tags"]:
        explanation_parts.append(
            "This is mainly a style or metadata-based discovery because the graph support comes from shared tags."
        )

    return " ".join(explanation_parts)

In [21]:
print(generate_explanation("Pearl Jam", "Soundgarden"))

Soundgarden was retrieved for Pearl Jam. In the vector retriever, this pair has a cosine similarity score of 0.741, based on artist embedding text. The graph retriever also found shared-member evidence: Matt Cameron connects Pearl Jam and Soundgarden. The graph also found shared tag evidence, including: alternative rock, grunge, hard rock, usa. The hybrid retriever combines vector similarity with graph evidence, giving this pair a hybrid score of 0.998. This is a relationship-based discovery because the connection is supported by explicit historical graph evidence.


In [22]:
print(generate_explanation("Nirvana", "Soundgarden"))

Soundgarden was retrieved for Nirvana. In the vector retriever, this pair has a cosine similarity score of 0.712, based on artist embedding text. The graph retriever also found shared-member evidence: Jason Everman connects Nirvana and Soundgarden. The graph also found shared tag evidence, including: alternative metal, alternative rock, american, grunge, sludge metal. The hybrid retriever combines vector similarity with graph evidence, giving this pair a hybrid score of 0.998. This is a relationship-based discovery because the connection is supported by explicit historical graph evidence.


In [23]:
print(generate_explanation("The Smashing Pumpkins", "Blur"))

Blur was retrieved for The Smashing Pumpkins. In the vector retriever, this pair has a cosine similarity score of 0.697, based on artist embedding text. The graph also found shared tag evidence, including: alternative rock, grunge, rock, shoegaze, neo-psychedelia. The hybrid retriever combines vector similarity with graph evidence, giving this pair a hybrid score of 0.797. This is mainly a style or metadata-based discovery because the graph support comes from shared tags.


In [24]:
print(generate_explanation("Soundgarden", "Blur"))

Blur was retrieved for Soundgarden. In the vector retriever, this pair has a cosine similarity score of 0.684, based on artist embedding text. The hybrid retriever combines vector similarity with graph evidence, giving this pair a hybrid score of 0.684.


In [25]:
print(generate_explanation("Blur", "Soundgarden"))

Soundgarden was retrieved for Blur. This pair does not appear in the saved top-k vector, graph, or hybrid retrieval results for this query direction. This does not necessarily mean the artists have no similarity or relationship; it only means the pair was not retrieved in the current saved top-k results.


In [26]:
example_pairs = [
    ("Pearl Jam", "Soundgarden"),
    ("Nirvana", "Soundgarden"),
    ("Blur", "Pearl Jam"),
]

explanation_rows = []

for query_artist, retrieved_artist in example_pairs:
    explanation_rows.append({
        "query_artist": query_artist,
        "retrieved_artist": retrieved_artist,
        "explanation": generate_explanation(query_artist, retrieved_artist)
    })

explanation_examples = pd.DataFrame(explanation_rows)

explanation_examples

,query_artist,retrieved_artist,explanation
0,Pearl Jam,Soundgarden,Soundgarden was retrieved for Pearl Jam. In th...
1,Nirvana,Soundgarden,Soundgarden was retrieved for Nirvana. In the ...
2,Blur,Pearl Jam,Pearl Jam was retrieved for Blur. In the vecto...


In [27]:
out_path = PROJECT_ROOT / "data" / "processed" / "explanation_examples.csv"

explanation_examples.to_csv(out_path, index=False)

out_path

WindowsPath('c:/uni/seriousism/reminisGraph/data/processed/explanation_examples.csv')

In [28]:
pd.read_csv(out_path).head()

,query_artist,retrieved_artist,explanation
0,Pearl Jam,Soundgarden,Soundgarden was retrieved for Pearl Jam. In th...
1,Nirvana,Soundgarden,Soundgarden was retrieved for Nirvana. In the ...
2,Blur,Pearl Jam,Pearl Jam was retrieved for Blur. In the vecto...


#### Phase 10B Conclusion

This notebook builds a baseline explanation layer for the retrieval system.

The explanation layer does not retrieve new artists. Instead, it explains results produced by the vector, graph, and hybrid retrievers.

The explanations combine vector similarity scores, graph evidence, and hybrid scores into readable summaries.

Important examples include Pearl Jam and Soundgarden, which are connected through Matt Cameron, and Nirvana and Soundgarden, which are connected through Jason Everman. These examples show how graph evidence can explain relationship-based music discovery.

This baseline uses template-based explanations for now. A future version can use an LLM to rewrite these structured explanations in a more natural style, while still keeping retrieval grounded in the existing retriever outputs.